# Week 2.2: Image Segmentation & License Plate Pipeline

**Computer Vision Fundamentals** — Temasek Poly IIT AAI

*(Student copy — fill in the code where you see **Task** or placeholder comments. Use the teacher copy for full solutions.)*

This notebook covers **image segmentation**: thresholding, edge detection, contour detection, and perspective correction. We combine these into a complete **license plate recognition pipeline**.

---
## 2.2.1 Why Segmentation?

In Week 2.1 we learned to **transform** images — resize, rotate, blur. But those operations assume we already know *what* to transform. In practice we first need to **find** the region of interest.

**Image segmentation** is the process of dividing an image into meaningful regions: separating foreground from background, isolating objects, or finding boundaries.

### The license plate pipeline

In this notebook we build a complete pipeline:
1. **Threshold** — convert to binary (foreground / background).
2. **Detect edges** — find boundaries of objects.
3. **Find contours** — extract closed shapes.
4. **Filter contours** — pick the quadrilateral that looks like a plate.
5. **Perspective-correct** — unwarp the plate to a flat rectangle.
6. **Segment characters** — isolate individual letters/digits.

Each step builds on the previous one. By the end you will have a working pipeline that takes a photo of a car and extracts individual characters from the license plate.

---
## 2.2.2 Setup: Imports and Loading Images

Same libraries as before. We load two images:
- `data/car.jpg` — a car photo with a visible license plate (we will detect and correct it).
- `data/cropped_plate.jpg` — a pre-cropped plate for practising thresholding and character segmentation.

**Task:** Write the import statements, define `bgr_to_rgb`, load both images, and display them side by side.

In [ ]:
# Import cv2, numpy as np, matplotlib.pyplot as plt, matplotlib.gridspec as gridspec

# Define bgr_to_rgb(img) that converts BGR to RGB using cv2.cvtColor

In [ ]:
# Load data/car.jpg into 'car' and data/cropped_plate.jpg into 'plate_crop'
# using cv2.imread()

# Print shape and dtype for each image
# Display both side by side using plt.subplots(1, 2)

---
## 2.2.3 Thresholding

**Thresholding** converts a grayscale image to **binary** (black and white). Every pixel becomes either 0 or 255 based on whether it is above or below a threshold value.

### Why threshold?
- Simplifies the image to foreground (white) vs background (black).
- Required before contour detection (`cv2.findContours` expects a binary image).
- Makes character isolation much easier.

### Types of thresholding

| Method | Syntax | How it works |
|---|---|---|
| **Binary** | `cv2.threshold(gray, thresh, 255, cv2.THRESH_BINARY)` | Pixel > thresh → 255, else → 0 |
| **Binary inverse** | `cv2.threshold(gray, thresh, 255, cv2.THRESH_BINARY_INV)` | Pixel > thresh → 0, else → 255 |
| **Otsu’s** | `cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)` | Automatically finds the best threshold |

`cv2.threshold` returns **two values**: `(threshold_value, binary_image)`. With Otsu, the first return tells you the computed threshold.

**Task:** Convert the cropped plate to grayscale, then apply binary thresholding at 100, binary inverse at 100, and Otsu’s method. Display all four (grayscale + three results).

In [ ]:
# Convert plate_crop to grayscale: gray_plate = cv2.cvtColor(...)

# Binary threshold at 100: _, binary = cv2.threshold(gray_plate, 100, 255, cv2.THRESH_BINARY)

# Binary inverse at 100: _, binary_inv = cv2.threshold(...)

# Otsu's method: otsu_val, binary_otsu = cv2.threshold(gray_plate, 0, 255, ...)
# Print the Otsu threshold value

# Display all four in plt.subplots(1, 4) with cmap='gray'

---
## 2.2.4 Morphological Operations

After thresholding, binary images often have **noise** (small white dots in the background) or **gaps** (small holes inside characters). **Morphological operations** clean these up by sliding a small shape (a **kernel** or **structuring element**) across the image.

### The four main operations

| Operation | Function | Effect | Use case |
|---|---|---|---|
| **Erosion** | `cv2.erode(img, kernel, iterations)` | Shrinks white regions | Remove thin noise, separate touching characters |
| **Dilation** | `cv2.dilate(img, kernel, iterations)` | Expands white regions | Connect broken parts, fill small gaps |
| **Opening** | `cv2.morphologyEx(img, cv2.MORPH_OPEN, kernel)` | Erosion → dilation | Remove noise while preserving shape size |
| **Closing** | `cv2.morphologyEx(img, cv2.MORPH_CLOSE, kernel)` | Dilation → erosion | Fill holes while preserving shape size |

The **kernel** is typically a small square of ones, e.g. `np.ones((3, 3), np.uint8)`. Larger kernels or more iterations = stronger effect.

**Task:** Apply erosion, dilation, opening, and closing to the thresholded plate. Display all five (binary + four results).

In [ ]:
# Threshold the plate: _, plate_bin = cv2.threshold(gray_plate, 100, 255, cv2.THRESH_BINARY)

# Create a 3x3 kernel: kernel = np.ones((3, 3), np.uint8)

# Apply erosion:  cv2.erode(plate_bin, kernel, iterations=1)
# Apply dilation: cv2.dilate(plate_bin, kernel, iterations=1)
# Apply opening:  cv2.morphologyEx(plate_bin, cv2.MORPH_OPEN, kernel)
# Apply closing:  cv2.morphologyEx(plate_bin, cv2.MORPH_CLOSE, kernel)

# Display all five in plt.subplots(1, 5) with cmap='gray'

---
## 2.2.5 Edge Detection with Canny

Edges are boundaries where pixel intensity changes sharply. The **Canny edge detector** is the most widely used edge detection method in CV.

### How Canny works (high level)
1. **Gaussian blur** — reduces noise (applied internally by Canny, but pre-blurring often helps).
2. **Gradient computation** — finds the intensity change in the x and y directions using Sobel filters.
3. **Non-maximum suppression** — thins edges to 1-pixel width by keeping only local maxima in the gradient direction.
4. **Hysteresis thresholding** with two thresholds:
   - Gradient > `high` → **strong edge** (definitely kept).
   - Gradient < `low` → **not an edge** (discarded).
   - Between `low` and `high` → **weak edge** (kept only if connected to a strong edge).

```python
edges = cv2.Canny(gray, low_threshold, high_threshold)
```

**Guideline:** Set `high ≈ 2–3× low`. Typical starting values: `low=50, high=150`.

**Task:** Apply Canny to a grayscale, blurred version of the car image with different threshold pairs. Display the original alongside the edge results.

In [ ]:
# Convert car to grayscale: gray_car = cv2.cvtColor(car, cv2.COLOR_BGR2GRAY)
# Apply Gaussian blur: gray_car = cv2.GaussianBlur(gray_car, (5, 5), 0)

# Apply Canny with different thresholds:
#   edges_low  = cv2.Canny(gray_car, 30, 100)
#   edges_mid  = cv2.Canny(gray_car, 75, 200)
#   edges_high = cv2.Canny(gray_car, 150, 300)

# Display original + three edge images in plt.subplots(1, 4) with cmap='gray'

---
## 2.2.6 Contour Detection and Drawing

A **contour** is a curve joining all continuous points along a boundary that have the same intensity. `cv2.findContours` finds these curves in a binary or edge image.

```python
contours, hierarchy = cv2.findContours(binary_img, mode, method)
```

### Retrieval modes (`mode`)
| Mode | Description |
|---|---|
| `cv2.RETR_EXTERNAL` | Only the outermost contours (ignores nested ones) |
| `cv2.RETR_LIST` | All contours as a flat list (no parent-child info) |
| `cv2.RETR_TREE` | Full hierarchy with parent-child relationships |

### Approximation methods (`method`)
| Method | Description |
|---|---|
| `cv2.CHAIN_APPROX_SIMPLE` | Compresses straight segments to their endpoints (saves memory) |
| `cv2.CHAIN_APPROX_NONE` | Stores every point on the contour |

### Drawing contours
```python
cv2.drawContours(image, contours, contourIdx, color, thickness)
```
Use `contourIdx = -1` to draw all contours at once.

### Useful contour properties
- **Area:** `cv2.contourArea(c)` — number of pixels inside.
- **Perimeter:** `cv2.arcLength(c, closed=True)` — length of the boundary.
- **Bounding rectangle:** `x, y, w, h = cv2.boundingRect(c)` — smallest upright rectangle containing the contour.

**Task:** Find contours in the Canny edge image of the car. Draw them on a copy. Print the total count.

In [ ]:
# Find contours in edges_mid using cv2.findContours with RETR_LIST and CHAIN_APPROX_SIMPLE
# Print the total number of contours found

# Draw all contours on a copy of car: cv2.drawContours(car_copy, contours, -1, (0,255,0), 2)

# Display edges and contour-annotated image in plt.subplots(1, 2)

---
## 2.2.7 Contour Filtering and Polygon Approximation

Most contours are irrelevant — edges of trees, shadows, road markings. We need to **filter** for the license plate.

### Polygon approximation with `cv2.approxPolyDP`

The **Douglas-Peucker algorithm** simplifies a contour to a polygon with fewer vertices:

```python
epsilon = factor * cv2.arcLength(contour, True)
approx = cv2.approxPolyDP(contour, epsilon, True)
```

- **`epsilon`** controls how much simplification: smaller ε = more vertices (closer to original shape), larger ε = fewer vertices (more aggressive simplification).
- A common default is `factor = 0.02` (2% of the perimeter).

### Strategy for finding the plate
1. Approximate each contour to a polygon.
2. Keep only **quadrilaterals** (polygons with exactly 4 vertices) — a license plate is rectangular.
3. Optionally filter by contour area or aspect ratio to reject false positives.

**Task:** Loop through the contours, approximate each one, and collect those with exactly 4 vertices. Draw the quadrilaterals on the image.

In [ ]:
# Loop through contours, approximate each with cv2.approxPolyDP
# Collect those with exactly 4 vertices into a list called 'quads'
# quads = []
# for c in contours:
#     peri = cv2.arcLength(c, True)
#     approx = cv2.approxPolyDP(c, 0.02 * peri, True)
#     if len(approx) == 4: ...

# Print how many quadrilaterals were found
# Draw quads on a copy of car and display in plt.subplots(1, 2):
#   left = all contours, right = quads only

---
## 2.2.8 Perspective Transform: Mathematics and Intuition

When a camera views a flat object (like a license plate) at an angle, the rectangle appears as a trapezoid or general quadrilateral. A **perspective transform** (or **homography**) maps the distorted quad back to a rectangle.

### Homogeneous coordinates

In regular 2D coordinates a point is $(x, y)$. In **homogeneous coordinates** we add a third value: $(x, y, 1)$.

After a perspective transform the result is $(x', y', w')$, and the actual 2D point is:

$$\left(\frac{x'}{w'},\; \frac{y'}{w'}\right)$$

This division by $w'$ is what creates the perspective effect — things farther away appear smaller.

> **Note on $w'$:** The output of the matrix multiplication is three numbers $(x', y', w')$, but the final destination point is **2D**, not 3D. The value $w'$ is not part of the output coordinate — it is only a divisor used to convert from homogeneous coordinates (3 numbers) back to regular 2D coordinates (2 numbers). The flow is:
> 1. Start with source point $(x, y)$ — regular 2D.
> 2. Write in homogeneous form: $(x, y, 1)$ — now 3 numbers.
> 3. Multiply by $H$: get $(x', y', w')$ — still 3 numbers, still homogeneous.
> 4. Divide out $w'$: get $\left(\frac{x'}{w'},\; \frac{y'}{w'}\right)$ — back to regular 2D.
>
> The third coordinate $w'$ exists only as an intermediate computation artifact. It is the mechanism that *enables* non-uniform scaling across the image, but it is discarded once the division is done.

### The 3×3 homography matrix

A perspective transform is described by a 3×3 matrix $H$ that maps source points to destination points:

$$\begin{bmatrix} x' \\\\ y' \\\\ w' \end{bmatrix} = \begin{bmatrix} h_{00} & h_{01} & h_{02} \\\\ h_{10} & h_{11} & h_{12} \\\\ h_{20} & h_{21} & 1 \end{bmatrix} \begin{bmatrix} x \\\\ y \\\\ 1 \end{bmatrix}$$

### What each entry in H represents

| Entry / group | Role | Effect on the output |
|---|---|---|
| $h_{00}$ | Scale in $x$ | Stretches or compresses the image horizontally |
| $h_{11}$ | Scale in $y$ | Stretches or compresses the image vertically |
| $h_{01}$ | Shear in $x$ | Tilts the image horizontally (like italicising text) |
| $h_{10}$ | Shear in $y$ | Tilts the image vertically |
| $h_{02}$ | Translation in $x$ | Shifts the image left or right |
| $h_{12}$ | Translation in $y$ | Shifts the image up or down |
| $h_{20}$ | Perspective in $x$ | Makes $w'$ depend on $x$ — objects further right appear different in size |
| $h_{21}$ | Perspective in $y$ | Makes $w'$ depend on $y$ — objects further down appear different in size |

- The top-left 2×2 block ($h_{00}, h_{01}, h_{10}, h_{11}$) handles **rotation, scaling, and shearing** — the affine part.
- The rightmost column ($h_{02}, h_{12}$) handles **translation**.
- The bottom row ($h_{20}, h_{21}$) is what makes this a **perspective** (projective) transform rather than just affine. These entries cause $w'$ to vary across the image, so the division by $w'$ produces non-uniform scaling — exactly the "things farther away look smaller" effect.

We fix $h_{22} = 1$ (the matrix is defined up to scale), leaving **8 unknowns**.

### Intuition: what $h_{20}$ and $h_{21}$ actually do

The six affine entries ($h_{00}$ through $h_{12}$) are fairly intuitive — they scale, rotate, shear, and translate. But $h_{20}$ and $h_{21}$ deserve more attention because they are what make a perspective transform *perspective*.

Recall the denominator that every pixel gets divided by:

$$w' = h_{20}\,x + h_{21}\,y + 1$$

This is a **linear function of pixel position**. It defines a "divisor" that varies smoothly across the image.

**Example — $h_{20} \neq 0$ (perspective in $x$):**

Suppose $h_{20} = 0.001$ and $h_{21} = 0$. Then:
- At the **left** edge ($x = 0$): $w' = 0.001 \times 0 + 1 = 1$ — coordinates unchanged.
- At the **right** edge ($x = 1000$): $w' = 0.001 \times 1000 + 1 = 2$ — coordinates get **divided by 2**, so the right side is compressed to half its size.

The right side of the image gets **squished** while the left side stays normal. This is exactly what happens when you look at a road going off to the right — the far end looks compressed. A negative $h_{20}$ would compress the left side instead.

**Example — $h_{21} \neq 0$ (perspective in $y$):**

Same idea but vertically. Suppose $h_{21} = 0.002$ and $h_{20} = 0$:
- At the **top** ($y = 0$): $w' = 1$ — normal size.
- At the **bottom** ($y = 500$): $w' = 0.002 \times 500 + 1 = 2$ — bottom half compressed.

Think of looking down a tall building — the base looks normal but the far end appears to shrink.

**Why this creates the trapezoid effect:**

A rectangle has parallel sides with equal lengths. But when one side gets divided by a larger $w'$ than the opposite side, that side becomes shorter. Parallel lines stop being parallel — a rectangle becomes a trapezoid. That is perspective distortion.

**The license plate example:**

When the camera sees the plate at an angle, one side of the plate is physically closer to the camera than the other. The close side appears wider, the far side appears narrower. The homography **reverses** this: it applies a compensating $w'$ divisor that shrinks the "too wide" side and expands the "too narrow" side, restoring the rectangle.

**Contrast: when $h_{20} = h_{21} = 0$:**

When both are zero, $w' = 1$ everywhere. Every pixel gets divided by the same constant. No part of the image shrinks more or less than any other. You get an **affine** transform — it can rotate, scale, shear, and translate, but parallel lines stay parallel. It cannot fix a trapezoid back into a rectangle.

| | What it controls | Real-world analogy |
|---|---|---|
| $h_{20}$ | How much the divisor grows as you move **right** | Viewing the plate from the left side — right edge is farther away |
| $h_{21}$ | How much the divisor grows as you move **down** | Viewing the plate from above — bottom edge is farther away |
| Both zero | No perspective, just affine | Viewing the plate perfectly head-on |

In summary, $h_{20}$ and $h_{21}$ encode *the camera's viewing angle relative to the flat surface*. Everything else in $H$ (scale, rotation, translation, shear) could be done without them. These two entries are what make perspective correction possible.

### Expanding the matrix multiplication

Writing out each row of the matrix multiplication gives three equations:

$$x' = h_{00}\,x + h_{01}\,y + h_{02}$$
$$y' = h_{10}\,x + h_{11}\,y + h_{12}$$
$$w' = h_{20}\,x + h_{21}\,y + 1$$

The actual destination coordinates $(u, v)$ are obtained by dividing by $w'$:

$$u = \frac{x'}{w'} = \frac{h_{00}\,x + h_{01}\,y + h_{02}}{h_{20}\,x + h_{21}\,y + 1}$$

$$v = \frac{y'}{w'} = \frac{h_{10}\,x + h_{11}\,y + h_{12}}{h_{20}\,x + h_{21}\,y + 1}$$

> **Key insight:** The denominator $w' = h_{20}\,x + h_{21}\,y + 1$ changes with each pixel's position. This position-dependent divisor is what creates the perspective "foreshortening" effect. For an affine transform, $h_{20} = h_{21} = 0$, so $w' = 1$ everywhere — no perspective distortion.

### From point correspondences to a linear system

We know a source point $(x, y)$ maps to a destination point $(u, v)$. Cross-multiplying to eliminate the fraction:

$$u\,(h_{20}\,x + h_{21}\,y + 1) = h_{00}\,x + h_{01}\,y + h_{02}$$
$$v\,(h_{20}\,x + h_{21}\,y + 1) = h_{10}\,x + h_{11}\,y + h_{12}$$

Rearranging so that $u$ and $v$ (known values) move to the right-hand side:

$$h_{00}\,x + h_{01}\,y + h_{02} - u\,h_{20}\,x - u\,h_{21}\,y = u$$
$$h_{10}\,x + h_{11}\,y + h_{12} - v\,h_{20}\,x - v\,h_{21}\,y = v$$

Each point pair gives **2 linear equations** in the 8 unknowns ($h_{00}, h_{01}, h_{02}, h_{10}, h_{11}, h_{12}, h_{20}, h_{21}$).

### Why exactly 4 point correspondences?

With 8 unknowns we need $8 / 2 = $ **4 point pairs** — exactly the 4 corners of our quadrilateral mapped to the 4 corners of the output rectangle.

### The 8×8 linear system

Label the four source corners $(x_1, y_1), \ldots, (x_4, y_4)$ and their destinations $(u_1, v_1), \ldots, (u_4, v_4)$. Stacking all 8 equations:

$$\begin{bmatrix} x_1 & y_1 & 1 & 0 & 0 & 0 & -u_1 x_1 & -u_1 y_1 \\\\ 0 & 0 & 0 & x_1 & y_1 & 1 & -v_1 x_1 & -v_1 y_1 \\\\ x_2 & y_2 & 1 & 0 & 0 & 0 & -u_2 x_2 & -u_2 y_2 \\\\ 0 & 0 & 0 & x_2 & y_2 & 1 & -v_2 x_2 & -v_2 y_2 \\\\ x_3 & y_3 & 1 & 0 & 0 & 0 & -u_3 x_3 & -u_3 y_3 \\\\ 0 & 0 & 0 & x_3 & y_3 & 1 & -v_3 x_3 & -v_3 y_3 \\\\ x_4 & y_4 & 1 & 0 & 0 & 0 & -u_4 x_4 & -u_4 y_4 \\\\ 0 & 0 & 0 & x_4 & y_4 & 1 & -v_4 x_4 & -v_4 y_4 \end{bmatrix} \begin{bmatrix} h_{00} \\\\ h_{01} \\\\ h_{02} \\\\ h_{10} \\\\ h_{11} \\\\ h_{12} \\\\ h_{20} \\\\ h_{21} \end{bmatrix} = \begin{bmatrix} u_1 \\\\ v_1 \\\\ u_2 \\\\ v_2 \\\\ u_3 \\\\ v_3 \\\\ u_4 \\\\ v_4 \end{bmatrix}$$

Or compactly: $A\mathbf{h} = \mathbf{b}$, solved as $\mathbf{h} = A^{-1}\mathbf{b}$.

**This is exactly what `cv2.getPerspectiveTransform(src, dst)` does internally:**
1. Takes your 4 source points (`src`) and 4 destination points (`dst`).
2. Builds the 8×8 matrix $A$ and vector $\mathbf{b}$ shown above.
3. Solves the linear system for the 8 unknowns.
4. Returns the 3×3 matrix $H$ (with $h_{22} = 1$).

### Ordering the 4 corners — `order_quad_points`

`cv2.getPerspectiveTransform` requires the source and destination points in the **same consistent order**. We use: **[top-left, top-right, bottom-right, bottom-left]**.

The trick uses two simple properties of image coordinates (where $y$ increases downward):

| Property | Formula | Identifies |
|---|---|---|
| **Sum** | $x + y$ | **Smallest** → top-left (nearest to origin); **Largest** → bottom-right |
| **Difference** | $y - x$ | **Smallest** → top-right (small $y$, large $x$); **Largest** → bottom-left (large $y$, small $x$) |

This works because the top-left corner has both small $x$ and small $y$ (so their sum is smallest), while the bottom-right has both large (so their sum is largest). The difference $y - x$ disambiguates the remaining two corners.

**Task:** Verify the ordering logic with the example points below. Compute the sum and difference for each point and confirm which corner it maps to.

In [ ]:
# Verify the ordering trick with example points:
# example_pts = np.array([[200, 50], [50, 20], [180, 150], [30, 130]], dtype=np.float32)
# Compute s = x + y (sum along axis=1) and diff = y - x for each point
# Print each point with its sum and diff
# Identify: TL = argmin(s), BR = argmax(s), TR = argmin(diff), BL = argmax(diff)

---
## 2.2.9 Perspective Correction with OpenCV

Now we apply the theory: take the quadrilateral detected on the car image, order its corners, compute the homography, and warp the plate to a flat rectangle.

```python
M = cv2.getPerspectiveTransform(src_4pts, dst_4pts)   # 3×3 homography
warped = cv2.warpPerspective(img, M, (width, height))
```

**Task:** Use `order_quad_points` on the detected quad, define the destination rectangle, compute the perspective transform, and warp the plate.

In [ ]:
def order_quad_points(pts):
    """Order quad corners as [top-left, top-right, bottom-right, bottom-left].
    Uses the sum (x+y) and difference (y-x) trick."""
    pts = np.squeeze(pts).astype(np.float32)
    # TODO: compute s = pts.sum(axis=1) and diff = np.diff(pts, axis=1)
    # TODO: tl = pts[np.argmin(s)], br = pts[np.argmax(s)]
    # TODO: tr = pts[np.argmin(diff)], bl = pts[np.argmax(diff)]
    # TODO: return np.array([tl, tr, br, bl], dtype=np.float32)
    pass

# Pick the largest quad by contour area: quad = max(quads, key=cv2.contourArea)
# Order its corners: src = order_quad_points(quad)

# Define the destination rectangle (W=600, H=200):
#   dst = np.float32([[0, 0], [W-1, 0], [W-1, H-1], [0, H-1]])

# Compute homography: M = cv2.getPerspectiveTransform(src, dst)
# Warp: warped = cv2.warpPerspective(car, M, (W, H))

# Print the source corners and the homography matrix
# Display the car with quad overlay and the warped plate in plt.subplots(1, 2)

---
## 2.2.10 Character Segmentation with Contours

With the plate unwarped to a flat rectangle, we can segment individual characters:

1. **Convert to grayscale**.
2. **Threshold** — use `THRESH_BINARY_INV` so characters become white (foreground) on black.
3. **Morphological cleanup** — erosion to separate touching characters.
4. **Find contours** of each character blob.
5. **Filter by aspect ratio** — characters are typically taller than wide (`h/w` between 1 and 3).
6. **Sort left to right** by the bounding rectangle’s x-coordinate.

**Task:** Threshold the warped plate, find character contours, filter by aspect ratio, sort left to right, and display each character in its own subplot.

In [ ]:
# Step 1-2: Convert warped to grayscale, threshold with THRESH_BINARY_INV at 100
# gray_warped = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)
# _, char_bin = cv2.threshold(gray_warped, 100, 255, cv2.THRESH_BINARY_INV)

# Step 3: Erode to clean up — kernel = np.ones((3, 3), np.uint8), iterations=2

# Display the thresholded plate

In [ ]:
# Step 4: Find contours with cv2.findContours using RETR_EXTERNAL

# Step 5: Filter — for each contour, get boundingRect, compute h/w ratio
# Keep characters where 1 <= ratio <= 3 and h > 10
# Store as list of (x, y, w, h) tuples

# Step 6: Sort the list by x coordinate (left to right)

# Display each character in its own subplot using gridspec or plt.subplots

---
## 2.2.11 In-class Application: Full License Plate Pipeline

Let’s put everything together into a single, clean pipeline function. Given a car image, the function:
1. Converts to grayscale and blurs.
2. Runs Canny edge detection.
3. Finds contours and filters for quadrilaterals.
4. Perspective-corrects the largest quad.
5. Thresholds and segments characters.

**Task:** Complete the `extract_plate_chars` function below by filling in each step. Then run it on the car image and display the results.

In [ ]:
def extract_plate_chars(img, canny_low=75, canny_high=200, char_thresh=100):
    """Full pipeline: car image -> list of character images."""

    # 1. Grayscale + Gaussian blur (5, 5)

    # 2. Canny edge detection with canny_low and canny_high

    # 3. Find contours -> approximate polygons -> filter for quads (4 vertices)
    #    Return empty list if no quads found

    # 4. Perspective-correct the largest quad (by area)
    #    Use order_quad_points, getPerspectiveTransform, warpPerspective
    #    Output size: 600 x 200

    # 5. Threshold the warped plate (BINARY_INV) and erode
    #    Find character contours, filter by aspect ratio, sort left to right

    pass  # Replace with your implementation

# Run the full pipeline on car
# chars, plate_img = extract_plate_chars(car)
# print(f"Extracted {len(chars)} characters")

# Display: input image, corrected plate with bounding boxes, and individual characters

---
## 2.2.12 Summary

| Technique | Key function(s) | Notes |
|---|---|---|
| **Thresholding** | `cv2.threshold(gray, thresh, 255, type)` | Binary, binary inverse, Otsu. Otsu finds the threshold automatically. |
| **Morphology** | `cv2.erode`, `cv2.dilate`, `cv2.morphologyEx` | Opening removes noise; closing fills gaps. Kernel size controls strength. |
| **Canny edges** | `cv2.Canny(gray, low, high)` | Two thresholds; `high ≈ 2–3× low`. Pre-blur for cleaner results. |
| **Contours** | `cv2.findContours(binary, mode, method)` | Returns a list of point arrays. Use `RETR_LIST` or `RETR_EXTERNAL`. |
| **Polygon approx** | `cv2.approxPolyDP(c, epsilon, True)` | Douglas-Peucker algorithm; `epsilon = 0.02 × perimeter` is a good default. |
| **Perspective transform** | `cv2.getPerspectiveTransform(src, dst)` | 4 point pairs → 3×3 homography. Order corners consistently. |
| **Warp** | `cv2.warpPerspective(img, M, (W, H))` | Applies the homography to produce the output image. |
| **Character segmentation** | Contours + bounding rects + aspect ratio filter | Sort left to right for reading order. |

### Pipeline recap
```
Car image → Grayscale → Blur → Canny edges
→ Find contours → Approximate polygons → Filter for quads
→ Order corners → Perspective warp → Flat plate
→ Threshold → Find character contours → Filter by aspect ratio
→ Sort left to right → Individual characters
```